# 03 · Señales deterministas — el cortafuegos de la verdad

**Objetivo:** demostrar que la UI puede decidir el estado final (`OK` / `ERROR` / `INDETERMINADO`) **sin leer la prosa del LLM**, usando solo:

- `stop_reason` (`end_turn` | `tool_use` | `max_tokens`)
- bloques `tool_use` y sus `tool_result` correspondientes
- el campo `is_error` y/o el shape del payload

Vamos a forzar tres escenarios — éxito, error 503 simulado, max_tokens — y verificar que el verdict se computa igual que en el prototipo (`prototype/src/lib/agenticLoop.ts`).

In [ ]:
import asyncio, nest_asyncio
from typing import Any
from dataclasses import dataclass, field
from claude_agent_sdk import (
    query, tool, create_sdk_mcp_server, ClaudeAgentOptions,
    AssistantMessage, ResultMessage,
)

nest_asyncio.apply()

## 1. Tool con escenario inyectable

La tool real respondería al core. Aquí inyectamos el escenario vía variable de módulo para reutilizar la misma definición.

In [ ]:
SCENARIO = 'success'  # success | timeout_503 | max_tokens

@tool(
    name='cancel_product',
    description='Cancela un producto financiero. Devuelve ticket o error 503.',
    input_schema={'account_id': str, 'product_id': str, 'product_type': str},
)
async def cancel_product(args: dict[str, Any]) -> dict[str, Any]:
    if SCENARIO == 'timeout_503':
        return {
            'content': [{
                'type': 'text',
                'text': 'HTTP 503 core_bancario_timeout — upstream sin respuesta en 1200ms',
            }],
            'is_error': True,
        }
    return {
        'content': [{
            'type': 'text',
            'text': f"ticket_id=CCL-2026-0073; state=CANCELLED; product_id={args['product_id']}",
        }]
    }

banking_server = create_sdk_mcp_server(name='banking', version='0.1.0', tools=[cancel_product])

## 2. Recolector de señales

Capturamos *exactamente* lo que la UI vería: `tool_use` blocks, `tool_result` blocks, `stop_reason`. La prosa la guardamos solo para auditoría — **no entra en el verdict**.

In [ ]:
@dataclass
class Signals:
    stop_reason: str | None = None
    tool_uses: list[dict] = field(default_factory=list)
    tool_results: dict[str, dict] = field(default_factory=dict)
    final_text: str = ''
    usage: dict = field(default_factory=dict)

async def collect_signals(prompt: str, options: ClaudeAgentOptions) -> Signals:
    sig = Signals()
    async for message in query(prompt=prompt, options=options):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                btype = getattr(block, 'type', None)
                if btype == 'tool_use':
                    sig.tool_uses.append({
                        'id': block.id, 'name': block.name, 'input': block.input,
                    })
                elif btype == 'tool_result':
                    sig.tool_results[block.tool_use_id] = {
                        'is_error': getattr(block, 'is_error', False),
                        'content': getattr(block, 'content', None),
                    }
                elif getattr(block, 'text', None):
                    sig.final_text += block.text
        if isinstance(message, ResultMessage):
            sig.stop_reason = message.stop_reason
            sig.usage = dict(message.usage or {})
    return sig

## 3. Decisión del verdict

Este es el equivalente Python de `decideVerdict()` en `prototype/src/lib/agenticLoop.ts`. Mismas reglas, mismo orden.

In [ ]:
def decide_verdict(sig: Signals) -> dict:
    if sig.stop_reason == 'max_tokens':
        return {'kind': 'warn', 'reason': 'loop cortado por max_tokens'}
    cancel_use = next((u for u in sig.tool_uses if u['name'] == 'cancel_product'), None)
    if not cancel_use:
        return {'kind': 'warn', 'reason': 'cancel_product nunca invocada'}
    res = sig.tool_results.get(cancel_use['id'])
    if res is None:
        return {'kind': 'err', 'reason': 'sin tool_result correspondiente'}
    if res['is_error']:
        return {'kind': 'err', 'reason': str(res['content'])[:160]}
    if sig.stop_reason != 'end_turn':
        return {'kind': 'warn', 'reason': f'stop_reason={sig.stop_reason} no terminal'}
    return {'kind': 'ok', 'evidence': res['content']}

## 4. Correr los tres escenarios

El último (`max_tokens`) lo simulamos forzando un `max_tokens` muy bajo: el modelo no alcanza a invocar la tool y sale con `stop_reason='max_tokens'`.

In [ ]:
PROMPT = (
    'Cancela la tarjeta de crédito TC-XX1234 de la cuenta ACC-9911. '
    'Reporta el ticket devuelto por cancel_product.'
)

BASE_OPTIONS = ClaudeAgentOptions(
    model='claude-opus-4-7',
    mcp_servers={'banking': banking_server},
    allowed_tools=['mcp__banking__cancel_product'],
    permission_mode='acceptEdits',
)

async def run_scenario(name: str, options: ClaudeAgentOptions):
    global SCENARIO
    SCENARIO = name
    sig = await collect_signals(PROMPT, options)
    verdict = decide_verdict(sig)
    print(f'\n=== escenario: {name} ===')
    print(f'stop_reason     : {sig.stop_reason}')
    print(f'tool_uses       : {[u["name"] for u in sig.tool_uses]}')
    print(f'tool_results    : {[(k, v["is_error"]) for k, v in sig.tool_results.items()]}')
    print(f'tokens          : in={sig.usage.get("input_tokens", 0)} out={sig.usage.get("output_tokens", 0)}')
    print(f'prosa final     : {sig.final_text[:140]}{"…" if len(sig.final_text) > 140 else ""}')
    print(f'>>> VERDICT     : {verdict}')
    return verdict

async def main():
    await run_scenario('success', BASE_OPTIONS)
    await run_scenario('timeout_503', BASE_OPTIONS)
    # Para max_tokens forzamos un límite agresivo.
    truncated = ClaudeAgentOptions(
        **{**BASE_OPTIONS.__dict__, 'max_thinking_tokens': 32, 'max_turns': 1},
    )
    await run_scenario('max_tokens', truncated)

asyncio.run(main())

## 5. Conclusión de la PoC

1. **El SDK expone las señales que necesitamos** — `stop_reason`, `tool_use`, `tool_result.is_error`, `usage`. Ninguna requiere parsear texto.
2. **`decide_verdict()` es portable** — la versión TS del prototipo y esta versión Python aplican la misma lógica. La regla *"no afirmar éxito sin ToolResult OK"* es código, no prompt-ingeniería.
3. **La prosa del modelo puede mentir** y aún así la UI/PoC reportan el estado real. Ese es el cortafuegos de la verdad enunciado en `propuesta_reformulada.md` §1.2.

**Próximo paso real (fuera de scope del taller):** intercambiar el mock `bankingCore` del prototipo por una llamada SSE a un backend que use exactamente este loop con el SDK, conservando el shape de `StreamEvent`.